# Ingredient Extraction Accuracy

**Purpose:** Run OCR + `normalize_ingredients()` on all 20 test product photos,
compare extracted ingredients against manually-annotated ground truth, and
report precision / recall / F1.

**Prerequisites:**
- Docker API running (default `http://localhost:8000`)
- 20 product JPEGs in `backend/test_photos/`
- Google service account at `backend/credentials/shelf-love-353bbeca17ab.json`
- Ground truth CSV at `backend/test_photos/ground_truth.csv` (for brand/product metadata)

In [1]:
import sys, os, json, time
from pathlib import Path

import requests
import pandas as pd
from IPython.display import display, Markdown

BACKEND_DIR = Path.cwd().parent
sys.path.insert(0, str(BACKEND_DIR))

from app.services.ingredient_normalization import normalize_ingredients
from app.services.vision import detect_text

In [2]:
API_BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:8000")

TEST_EMAIL = "ocr_test@example.com"
TEST_PASSWORD = "test123"
TEST_NAME = "OCR Test"

NOTEBOOK_DIR = Path.cwd()
BACKEND_DIR = NOTEBOOK_DIR.parent
IMAGE_DIR = BACKEND_DIR / "test_photos"
GT_PATH = IMAGE_DIR / "ground_truth.csv"
INGREDIENT_GT_PATH = IMAGE_DIR / "ingredient_ground_truth.json"

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
image_paths = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS])

print(f"API: {API_BASE_URL}")
print(f"Images dir: {IMAGE_DIR}")
print(f"Images found: {len(image_paths)}")
print(f"Ground truth CSV exists: {GT_PATH.exists()}")
print(f"Ingredient GT JSON exists: {INGREDIENT_GT_PATH.exists()}")

API: http://localhost:8000
Images dir: c:\Projects\cosmetic-expiry-scanner\backend\test_photos
Images found: 20
Ground truth CSV exists: True
Ingredient GT JSON exists: False


In [3]:
def login_or_register(email, password, name):
    r = requests.post(f"{API_BASE_URL}/auth/login", json={"email": email, "password": password})
    if r.status_code == 200:
        print(f"Logged in as {email}")
        return r.json()["access_token"]
    print(f"Login failed ({r.status_code}), registering...")
    r = requests.post(f"{API_BASE_URL}/auth/register", json={"email": email, "password": password, "name": name})
    if r.status_code == 201:
        print(f"Registered as {email}, logging in...")
        r = requests.post(f"{API_BASE_URL}/auth/login", json={"email": email, "password": password})
        if r.status_code == 200:
            return r.json()["access_token"]
        raise RuntimeError(f"Login after register failed: {r.status_code} {r.text}")
    raise RuntimeError(f"Auth failed: {r.status_code} {r.text}")

TOKEN = login_or_register(TEST_EMAIL, TEST_PASSWORD, TEST_NAME)
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

Login failed (401), registering...
Registered as ocr_test@example.com, logging in...


## 1. Upload images + OCR

Upload each photo to S3 via presigned URL, then call Google Vision directly
(same path as `vision.detect_text` in production).

In [4]:
def upload_and_ocr(image_path):
    filename = image_path.name
    r = requests.post(
        f"{API_BASE_URL}/uploads/presigned-url",
        json={"file_name": filename, "content_type": "image/jpeg"},
        headers=HEADERS,
    )
    r.raise_for_status()
    ud = r.json()

    with open(image_path, "rb") as f:
        data = f.read()
    r = requests.put(ud["upload_url"], data=data, headers={"Content-Type": "image/jpeg"})
    if r.status_code not in (200, 201, 204):
        raise RuntimeError(f"S3 upload failed: {r.status_code}")

    download_info = requests.get(
        f"{API_BASE_URL}/uploads/{ud['file_key']}/url", headers=HEADERS
    )
    download_info.raise_for_status()
    download_url = download_info.json()["download_url"]

    result = detect_text(download_url)
    return result["raw_text"]

In [5]:
ocr_cache = {}
results = []
print(f"Processing {len(image_paths)} images...\n")

for i, img_path in enumerate(image_paths, 1):
    print(f"[{i}/{len(image_paths)}] {img_path.name}")
    try:
        if img_path.name in ocr_cache:
            raw_text = ocr_cache[img_path.name]
            print(f"  cached: {len(raw_text)} chars")
        else:
            raw_text = upload_and_ocr(img_path)
            ocr_cache[img_path.name] = raw_text
            print(f"  OCR done: {len(raw_text)} chars")

        extracted = normalize_ingredients(raw_text)
        ingredient_names = [r["canonical_name"] for r in extracted if r["canonical_name"]]
        print(f"  Extracted {len(ingredient_names)} ingredients")

        results.append({
            "filename": img_path.name,
            "raw_ocr_text": raw_text,
            "ocr_length": len(raw_text),
            "extracted_names": ingredient_names,
            "n_extracted": len(ingredient_names),
            "success": True,
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "filename": img_path.name,
            "raw_ocr_text": "",
            "ocr_length": 0,
            "extracted_names": [],
            "n_extracted": 0,
            "success": False,
        })
    print()

ok = sum(1 for r in results if r["success"])
print(f"Done. {ok}/{len(results)} succeeded.")

Processing 20 images...

[1/20] 01.jpeg
  OCR done: 299 chars
  Extracted 0 ingredients

[2/20] 02.jpeg
  OCR done: 2693 chars
  Extracted 0 ingredients

[3/20] 03.jpeg
  OCR done: 747 chars
  Extracted 4 ingredients

[4/20] 04.jpeg
  OCR done: 227 chars
  Extracted 0 ingredients

[5/20] 05.jpeg
  OCR done: 1682 chars
  Extracted 9 ingredients

[6/20] 06.jpeg
  OCR done: 144 chars
  Extracted 0 ingredients

[7/20] 07.jpeg
  OCR done: 1641 chars
  Extracted 3 ingredients

[8/20] 08.jpeg
  OCR done: 345 chars
  Extracted 0 ingredients

[9/20] 09.jpeg
  OCR done: 433 chars
  Extracted 0 ingredients

[10/20] 10.jpeg
  OCR done: 17 chars
  Extracted 0 ingredients

[11/20] 11.jpeg
  OCR done: 395 chars
  Extracted 0 ingredients

[12/20] 12.jpeg
  OCR done: 11 chars
  Extracted 0 ingredients

[13/20] 13.jpeg
  OCR done: 1128 chars
  Extracted 3 ingredients

[14/20] 14.jpeg
  OCR done: 102 chars
  Extracted 0 ingredients

[15/20] 15.jpeg
  OCR done: 1524 chars
  Extracted 0 ingredients

[16/20

## 2. Review extracted ingredients

Run the cell below, then use the output to fill in `EXPECTED_INGREDIENTS`
in the next cell.

In [6]:
df_gt = pd.read_csv(GT_PATH)

df_results = pd.DataFrame(results)
df_results = df_results.merge(
    df_gt[["filename", "brand", "product_name", "label_side"]],
    on="filename", how="left",
)

for _, row in df_results.iterrows():
    print("=" * 60)
    print(f"{row['filename']}  |  {row['brand']}  |  {row['product_name']}  |  {row['label_side']}")
    print(f"OCR: {row['ocr_length']} chars  |  Extracted: {row['n_extracted']} ingredients")
    if row["extracted_names"]:
        print(f"  {row['extracted_names']}")
    else:
        print("  (none)")
    print()

01.jpeg  |  BIODERMA  |  Sensibio H2O Micellar Water  |  front
OCR: 299 chars  |  Extracted: 0 ingredients
  (none)

02.jpeg  |  BIODERMA  |  Sensibio H2O Micellar Water  |  back
OCR: 2693 chars  |  Extracted: 0 ingredients
  (none)

03.jpeg  |  Sesderma  |  DRYSES Deodorant Roll-On  |  back
OCR: 747 chars  |  Extracted: 4 ingredients
  ['Propylene Glycol', 'Acrylates/C10-30 Alkyl Acrylate Crosspolymer', 'Methylparaben', 'Fragrance']

04.jpeg  |  Sesderma  |  DRYSES Deodorant Roll-On  |  front
OCR: 227 chars  |  Extracted: 0 ingredients
  (none)

05.jpeg  |  L'Oreal Paris  |  Elvive Dream Lengths Shampoo  |  back
OCR: 1682 chars  |  Extracted: 9 ingredients
  ['Water', 'Sodium Laureth Sulfate', 'Cocamidopropyl Betaine', 'Dimethicone', 'Fragrance', 'Citric Acid', 'Butylene Glycol', 'Sodium Benzoate', 'Carbomer']

06.jpeg  |  L'Oreal Paris  |  Elvive Dream Lengths Shampoo  |  front
OCR: 144 chars  |  Extracted: 0 ingredients
  (none)

07.jpeg  |  Aveeno  |  Apple Cider Vinegar Shampoo  |

## 3. Ground truth

Fill in `EXPECTED_INGREDIENTS` below. For each photo:
- Use **canonical ingredient names** (lowercase, matching `dataset.json`)
- Set to `None` for photos with no ingredient panel (e.g. front labels)
- Leave as `[]` (empty list) for back labels where you expect ingredients but the parser found none

In [7]:
EXPECTED_INGREDIENTS = {
    "01.jpeg": None,
    "02.jpeg": [],
    "03.jpeg": ["Propylene Glycol", "Water", "Cyclopentasiloxane", "Acrylates/C10-30 Alkyl Acrylate Crosspolymer", "Methylparaben", "Fragrance", "Phenoxyethanol"],
    "04.jpeg": None,
    "05.jpeg": ["Water", "Sodium Laureth Sulfate", "Cocamidopropyl Betaine", "Dimethicone", "Fragrance", "Citric Acid", "Sodium Benzoate", "Carbomer", "Glycerin", "Salicylic Acid", "PEG-100 Stearate", "Niacinamide", "Panthenol", "Phenoxyethanol", "Titanium Dioxide"],
    "06.jpeg": None,
    "07.jpeg": ["Water", "Cocamidopropyl Betaine", "Propylene Glycol", "Citric Acid", "Sodium Benzoate", "Fragrance"],
    "08.jpeg": None,
    "09.jpeg": [],
    "10.jpeg": None,
    "11.jpeg": [],
    "12.jpeg": None,
    "13.jpeg": ["Water", "Glycerin", "Butylene Glycol", "Propylene Glycol", "Phenoxyethanol", "Propanediol", "Decyl Glucoside", "Panthenol", "Tocopherol", "Fragrance"],
    "14.jpeg": None,
    "15.jpeg": ["Titanium Dioxide", "Zinc Oxide"],
    "16.jpeg": None,
    "17.jpeg": ["Titanium Dioxide", "Zinc Oxide"],
    "18.jpeg": None,
    "19.jpeg": ["Water", "Fragrance", "Niacinamide", "Phenoxyethanol"],
    "20.jpeg": None,
}

filled = sum(1 for v in EXPECTED_INGREDIENTS.values() if v is not None)
print(f"Ground truth: {filled}/{len(EXPECTED_INGREDIENTS)} photos annotated")

Ground truth: 10/20 photos annotated


## 4. Accuracy metrics

In [8]:
def compute_metrics(extracted, expected):
    if expected is None:
        if not extracted:
            return {"tp": [], "fp": [], "fn": [], "precision": None, "recall": None, "f1": None, "skipped": True}
        return {"tp": [], "fp": list(extracted), "fn": [], "precision": 0.0, "recall": None, "f1": None, "skipped": False}
    ext_set = set(extracted)
    exp_set = set(expected)
    tp = sorted(ext_set & exp_set)
    fp = sorted(ext_set - exp_set)
    fn = sorted(exp_set - ext_set)
    precision = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) > 0 else None
    recall = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) > 0 else None
    f1 = (2 * precision * recall / (precision + recall)) if (precision is not None and recall is not None and precision + recall > 0) else None
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1, "skipped": False}

rows = []
for r in results:
    fn = r["filename"]
    expected = EXPECTED_INGREDIENTS.get(fn)
    m = compute_metrics(r["extracted_names"], expected)
    gt_row = df_gt[df_gt["filename"] == fn].iloc[0] if fn in df_gt["filename"].values else {}
    rows.append({
        "filename": fn,
        "brand": gt_row.get("brand", ""),
        "label_side": gt_row.get("label_side", ""),
        "n_extracted": r["n_extracted"],
        "n_expected": len(expected) if expected is not None else "N/A",
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "false_positives": m["fp"],
        "false_negatives": m["fn"],
        "skipped": m["skipped"],
    })

df_metrics = pd.DataFrame(rows)
display(df_metrics[["filename", "brand", "label_side", "n_extracted", "n_expected",
                    "precision", "recall", "f1"]])

,filename,brand,label_side,n_extracted,n_expected,precision,recall,f1
0,01.jpeg,BIODERMA,front,0,N/A,NaN,NaN,NaN
1,02.jpeg,BIODERMA,back,0,0,NaN,NaN,NaN
2,03.jpeg,Sesderma,back,4,7,1.000000,0.571429,0.727273
3,04.jpeg,Sesderma,front,0,N/A,NaN,NaN,NaN
4,05.jpeg,L'Oreal Paris,back,9,15,0.888889,0.533333,0.666667
5,06.jpeg,L'Oreal Paris,front,0,N/A,NaN,NaN,NaN
6,07.jpeg,Aveeno,back,3,6,1.000000,0.500000,0.666667
7,08.jpeg,Aveeno,front,0,N/A,NaN,NaN,NaN
8,09.jpeg,Natasha Denona,back,0,0,NaN,NaN,NaN
9,10.jpeg,Natasha Denona,front,0,N/A,NaN,NaN,NaN


In [9]:
evaluated = df_metrics[~df_metrics["skipped"]].copy()
no_ingredients_expected = df_metrics[df_metrics["skipped"]].copy()

print("=" * 60)
print("OVERALL METRICS")
print("=" * 60)

if len(evaluated) > 0:
    precisions = evaluated["precision"].dropna()
    recalls = evaluated["recall"].dropna()
    f1s = evaluated["f1"].dropna()
    print(f"  Photos with expected ingredients: {len(evaluated)}")
    print(f"  Macro Precision: {precisions.mean():.3f}" if len(precisions) else "  Macro Precision: N/A")
    print(f"  Macro Recall:    {recalls.mean():.3f}" if len(recalls) else "  Macro Recall: N/A")
    print(f"  Macro F1:        {f1s.mean():.3f}" if len(f1s) else "  Macro F1: N/A")

    print(f"\n  Perfect (F1=1.0):  {(f1s == 1.0).sum()} / {len(f1s)}")
    print(f"  Partial (0<F1<1):  {((f1s > 0) & (f1s < 1)).sum()} / {len(f1s)}")
    print(f"  Failed  (F1=0):    {(f1s == 0).sum()} / {len(f1s)}")
else:
    print("  No photos with expected ingredients annotated.")

print(f"\n  No-ingredient photos (expected=None): {len(no_ingredients_expected)}")
false_positives_on_empty = no_ingredients_expected[no_ingredients_expected["n_extracted"] > 0]
if len(false_positives_on_empty):
    print(f"  False positives on empty photos: {len(false_positives_on_empty)}")
    for _, r in false_positives_on_empty.iterrows():
        print(f"    {r['filename']}: extracted {r['n_extracted']} ingredients unexpectedly")
else:
    print(f"  False positives on empty photos: 0 (clean)")

OVERALL METRICS
  Photos with expected ingredients: 10
  Macro Precision: 0.778
  Macro Recall:    0.272
  Macro F1:        0.631

  Perfect (F1=1.0):  0 / 4
  Partial (0<F1<1):  4 / 4
  Failed  (F1=0):    0 / 4

  No-ingredient photos (expected=None): 10
  False positives on empty photos: 0 (clean)


In [10]:
print("=" * 60)
print("PER-PHOTO DETAIL")
print("=" * 60)

for _, row in df_metrics.iterrows():
    status = "SKIP" if row["skipped"] else ("OK" if row["f1"] == 1.0 else f"F1={row['f1']:.2f}" if row["f1"] is not None else "N/A")
    print(f"\n{row['filename']:>9}  {row['brand']:<20}  {row['label_side']:<5}  [{status}]")
    print(f"           Extracted: {row['n_extracted']}  |  Expected: {row['n_expected']}")
    if row["false_positives"]:
        print(f"           FP (not expected): {row['false_positives']}")
    if row["false_negatives"]:
        print(f"           FN (missed):       {row['false_negatives']}")

PER-PHOTO DETAIL

  01.jpeg  BIODERMA              front  [SKIP]
           Extracted: 0  |  Expected: N/A

  02.jpeg  BIODERMA              back   [F1=nan]
           Extracted: 0  |  Expected: 0

  03.jpeg  Sesderma              back   [F1=0.73]
           Extracted: 4  |  Expected: 7
           FN (missed):       ['Cyclopentasiloxane', 'Phenoxyethanol', 'Water']

  04.jpeg  Sesderma              front  [SKIP]
           Extracted: 0  |  Expected: N/A

  05.jpeg  L'Oreal Paris         back   [F1=0.67]
           Extracted: 9  |  Expected: 15
           FP (not expected): ['Butylene Glycol']
           FN (missed):       ['Glycerin', 'Niacinamide', 'PEG-100 Stearate', 'Panthenol', 'Phenoxyethanol', 'Salicylic Acid', 'Titanium Dioxide']

  06.jpeg  L'Oreal Paris         front  [SKIP]
           Extracted: 0  |  Expected: N/A

  07.jpeg  Aveeno                back   [F1=0.67]
           Extracted: 3  |  Expected: 6
           FN (missed):       ['Cocamidopropyl Betaine', 'Propylene Glyc

## 5. Save ground truth

In [11]:
with open(INGREDIENT_GT_PATH, "w", encoding="utf-8") as f:
    json.dump(EXPECTED_INGREDIENTS, f, indent=2, ensure_ascii=False)
print(f"Saved to {INGREDIENT_GT_PATH}")

Saved to c:\Projects\cosmetic-expiry-scanner\backend\test_photos\ingredient_ground_truth.json
